# Step 1 — Import everything we need


In [16]:
import os
from dotenv import load_dotenv  # to access the api keys

#Langchain
from langchain_community.document_loaders import TextLoader
#langchain_community is a package
#document_loaders is a class of the package
#TextLoader is a method


from langchain_text_splitters import RecursiveCharacterTextSplitter #(recursively breaks text into semantically meaningful chunks)
from langchain_community.embeddings import JinaEmbeddings

C:\Users\91639\AppData\Local\Temp\ipykernel_18536\2260703109.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [17]:
load_dotenv()

True

In [18]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")

ENV VAR LOADED


# LOADING OUR DATA

In [19]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")

# DATA INGESTION

In [20]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print("DATA LOADED")
print("="*40)
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

# LANGCHAIN DOCUMENT
Langchain processes everything in form of documents

DOCUMENTS :

PAGE CONTENT -- the actual data

METADATA - extra information about the data

In [21]:
len(documents)

1

In [22]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [23]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


# SPLITTING OUR DATA

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [25]:
len(chunks)

9

In [26]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [27]:
print(chunks[7])

page_content='7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.' metadata={'source': 'data\\hr_policy.txt'}


In [28]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


# EMBEDD OUR DATA

In [29]:
from langchain_community.embeddings import JinaEmbeddings
embeddings_model = JinaEmbeddings(model_name = "jina-embeddings-v2-base-en")
print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


### STORE DATA IN VECTOR DB

In [30]:
from langchain_community.vectorstores import FAISS 

vector_store = FAISS.from_documents(chunks , embeddings_model)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [31]:
# WE NEVER STORED OUR DATA

In [32]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



### TOOL

In [33]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

# DATA RETRIVAL

### LLM

In [34]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0   #creativity
)

llm.model_name

'openai/gpt-oss-120b'

In [35]:
tr = llm.invoke("Hey what is the leave policy")

In [36]:
tr.content

'Sure thing! Could you let me know which organization’s leave policy you’re interested in (e.g., your employer, a specific company, or a particular country’s labor regulations)? That’ll help me give you the most relevant information.'

### AI AGENT

3 --

LLM - BRAIN

TOOL - SUPER POWER

MEMORY - no memory

In [37]:
from langchain.agents import create_agent

In [38]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [39]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [40]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [41]:
response

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='f313ba28-277f-430a-b723-1175798f37aa'),
  AIMessage(content='I’m the HR assistant for **Acme Crop**. How can I help you today?', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". The assistant is a friendly HR assistant working for Acme Crop. Should answer that. No need to search policy. It\'s not about policy. So answer directly.'}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 208, 'total_tokens': 282, 'completion_time': 0.161640769, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.009548015, 'prompt_tokens_details': None, 'queue_time': 0.3484902, 'total_time': 0.171188784}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4140daa9c2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11201-947c-7991

### 

SYSTEM MESSAGE - HR ASSISNT

HUMAN MESSAGE - TELL ME ABOUT POLICIES

AI MESSAGE - HEY THESE ARE THEPLOICES

In [42]:
response["messages"][-2].content

'tell me which org you work for'

In [43]:
response["messages"][-1].content

'I’m the HR assistant for **Acme Crop**. How can I help you today?'